In [13]:
from math import floor, log10
import numpy as np
import pandas as pd

In [14]:
def read_pgc_cmt(filepath):
    df = pd.read_csv(filepath, sep='\s+', header=None,
                        names=['Year', 'Month', 'Day', 'Hour', 'Minute', 'Second',
                                 'Latitude', 'Longitude', 'DepthL', 'ML', 'DepthMT',
                                 'Misfit', 'Mxx', 'Myy', 'Mzz', 'Mxy', 'Mxz', 'Myz',
                                 'Mw', 'M0', 'Iso', 'CLVD', 'S1', 'D1', 'R1', 'S2', 'D2', 'R2',
                                 'Class', 'AzP', 'PlP', 'AzT', 'PlT', 'AzB', 'PlB'])

    return df

def get_exponent(m0):
    return floor(log10(m0))

In [36]:
pgc_cmt_df = read_pgc_cmt('/local/lyakuden/offshore_bc/bc-multiplets/gcmt/PGC_CMT.txt')
pgc_cmt_df.head()

,Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,DepthL,ML,...,S2,D2,R2,Class,AzP,PlP,AzT,PlT,AzB,PlB
0,1995,1,9,6,50,12.0,51.012,-130.790,6.0,4.5,...,153.66,64.04,-159.87,E1,12.22,31.89,105.47,5.21,203.72,57.58
1,1995,3,8,16,29,58.0,50.485,-130.246,12.0,4.6,...,137.08,83.17,169.30,E1,183.01,2.65,92.43,12.38,284.89,77.33
2,1995,4,23,9,29,49.0,50.453,-130.196,9.0,3.8,...,128.52,85.89,162.63,E1,175.39,9.20,82.88,15.14,295.61,72.16
3,1995,5,31,3,38,1.0,50.954,-130.676,12.0,4.9,...,144.12,80.88,176.35,E1,9.10,3.88,99.72,9.01,256.04,80.18
4,1995,9,12,22,44,23.0,51.209,-131.225,15.0,4.3,...,142.08,80.99,165.63,E1,188.74,3.59,97.68,16.50,290.64,73.09


In [37]:
# add column for string of YearMonthDayHourMinuteSecond with no decimals
pgc_cmt_df['Event'] = pgc_cmt_df.apply(lambda row: f"{int(row['Year']):04d}{int(row['Month']):02d}{int(row['Day']):02d}{int(row['Hour']):02d}{int(row['Minute']):02d}{int(row['Second']):02d}", axis=1)
pgc_cmt_df.head()

,Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,DepthL,ML,...,D2,R2,Class,AzP,PlP,AzT,PlT,AzB,PlB,Event
0,1995,1,9,6,50,12.0,51.012,-130.790,6.0,4.5,...,64.04,-159.87,E1,12.22,31.89,105.47,5.21,203.72,57.58,19950109065012
1,1995,3,8,16,29,58.0,50.485,-130.246,12.0,4.6,...,83.17,169.30,E1,183.01,2.65,92.43,12.38,284.89,77.33,19950308162958
2,1995,4,23,9,29,49.0,50.453,-130.196,9.0,3.8,...,85.89,162.63,E1,175.39,9.20,82.88,15.14,295.61,72.16,19950423092949
3,1995,5,31,3,38,1.0,50.954,-130.676,12.0,4.9,...,80.88,176.35,E1,9.10,3.88,99.72,9.01,256.04,80.18,19950531033801
4,1995,9,12,22,44,23.0,51.209,-131.225,15.0,4.3,...,80.99,165.63,E1,188.74,3.59,97.68,16.50,290.64,73.09,19950912224423


In [40]:
# make new dataframe to fit GMT format
pgc_cmt_gmt_df = pgc_cmt_df[['Longitude', 'Latitude', 'DepthMT', 
                                    'Mzz', 'Mxx', 'Myy', 
                                    'Mxz', 'Myz', 'Mxy', 'Event']].copy()

# add constant column of 20 for exponent, before event_id
pgc_cmt_gmt_df.insert(9, 'Exponent', 20)
pgc_cmt_gmt_df.insert(10, 'X', 'X')
pgc_cmt_gmt_df.insert(11, 'Y', 'Y')

pgc_cmt_gmt_df.head()

,Longitude,Latitude,DepthMT,Mzz,Mxx,Myy,Mxz,Myz,Mxy,Exponent,X,Y,Event
0,-130.790,51.012,6.0,27.09,61.81,-88.90,46.25,0.78,40.39,20,X,Y,19950109065012
1,-130.246,50.485,12.0,-4.39,99.34,-94.95,-3.72,-21.17,9.27,20,X,Y,19950308162958
2,-130.196,50.453,9.0,-4.27,95.38,-91.11,-18.86,-23.75,-19.28,20,X,Y,19950423092949
3,-130.676,50.954,12.0,-1.99,94.27,-92.28,9.27,-14.16,31.79,20,X,Y,19950531033801
4,-131.225,51.209,15.0,-7.68,95.67,-87.99,-2.53,-27.94,27.13,20,X,Y,19950912224423


In [41]:
# export to whitespace-delimited text file with no header and no index
pgc_cmt_gmt_df.to_csv('/local/lyakuden/offshore_bc/bc-multiplets/gcmt/PGC_all.txt', sep=' ', header=False, index=False)